# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsgluedi.transforms import *
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.5 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: dcf18057-81ad-41c2-aab5-b126289f4ebe
Applying the following default arguments:
--glue_kernel_version 1.0.5
--enable-glue-datacatalog true
Waiting for session dcf18057-81ad-41c2-aab5-b126289f4ebe to get into ready status...
Session dcf18057-81ad-41c2-aab5-b126289f4ebe ha

#### Example: Create a Static dataframe


In [2]:
input_df = spark.createDataFrame([(1,"jk", "1000.00"),(2, "pk","2000.00"),(3,"ak", "3000.00"), (4,"ck", "4000.00")],
    ["id","Name", "Account balance"],)
input_df.show()


+---+----+---------------+
| id|Name|Account balance|
+---+----+---------------+
|  1|  jk|        1000.00|
|  2|  pk|        2000.00|
|  3|  ak|        3000.00|
|  4|  ck|        4000.00|
+---+----+---------------+


#### Define Encryption function 
###### import boto
###### boto3 is the Amazon Web Services (AWS) Software Development Kit (SDK) for Python, which allows Python developers to write software that makes use of services like Amazon S3 and KMS etc

In [14]:
import boto3
def encrypt_text(text):
    kms_client = boto3.client('kms')
    key_id = 'b9e0f64f-9d90-4ccd-8a53-8d232c8c2218'
    #response1 = kms_client.encrypt(KeyId=key_id,Plaintext=text.encode())
   # print(response1)
    response = kms_client.encrypt(KeyId=key_id,Plaintext=text.encode())['CiphertextBlob']
    return response

#### Define Decryption function 

In [20]:
# decryption function
def decrypt_text(ciphertext):
    kms_client = boto3.client('kms')
    key_id = 'b9e0f64f-9d90-4ccd-8a53-8d232c8c2218'
    response2 = kms_client.decrypt(KeyId=key_id,CiphertextBlob=ciphertext)
    print(response2)
    response = kms_client.decrypt(KeyId=key_id,CiphertextBlob=ciphertext)['Plaintext'].decode('utf-8')
    return response

#### Testing Encryption & Decryption function 

In [21]:
text = '1000.00'
#testing the Encryption function
encrypted_text = encrypt_text(text)
print(f'Encrypted text: {encrypted_text}')

#testing the Decryption function
decrypted_text=decrypt_text(encrypted_text)
print(f'Plain text: {decrypted_text}')

Encrypted text: b'\x01\x02\x02\x00x\xe2*8\xf3\xe5\x08\xdb\xd3\xa9\x03\x1cuO\xb7\xfcab\xd0\xeez\xad\xa7\x90\xc8\xf8\x1e\xcf\r(\xbd\xaap\x01\x8c!y\xa1g\x16\xf2G\xb7\x10S=9\xd4\xd2\xb9\x00\x00\x00e0c\x06\t*\x86H\x86\xf7\r\x01\x07\x06\xa0V0T\x02\x01\x000O\x06\t*\x86H\x86\xf7\r\x01\x07\x010\x1e\x06\t`\x86H\x01e\x03\x04\x01.0\x11\x04\x0cH\x15\r\xe5\x9d;\x1e\xc8\xffE6K\x02\x01\x10\x80"\xef\x04\xda\r]\x86\x98/b(^\x91,I\xde\x96E\xb2v]\x13\xace\xb5\xc8X\x12\xaa\x8e\xa0\x9d;Tl'
{'KeyId': 'arn:aws:kms:ap-southeast-2:323619686659:key/b9e0f64f-9d90-4ccd-8a53-8d232c8c2218', 'Plaintext': b'1000.00', 'EncryptionAlgorithm': 'SYMMETRIC_DEFAULT', 'ResponseMetadata': {'RequestId': '7fc624d0-db89-4581-93be-898174b0a0d6', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '7fc624d0-db89-4581-93be-898174b0a0d6', 'cache-control': 'no-cache, no-store, must-revalidate, private', 'expires': '0', 'pragma': 'no-cache', 'date': 'Sun, 08 Sep 2024 17:25:55 GMT', 'content-type': 'application/x-amz-json-1.1', 'c

#### How to Convert a python function to pyspark udf (user define function)
##### PySpark UDF’s are similar to UDF on traditional databases. In PySpark, you create a function in a Python syntax and wrap it with PySpark SQL udf() or register it as udf and use it on DataFrame and SQL respectively.

In [22]:
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType
#Converting function to UDF 
encryptUDF = udf(encrypt_text, StringType())
decryptUDF = udf(decrypt_text, StringType())

#### How to Encrypt a dataframe column using UDF 

In [23]:
input_df.show()

+---+----+---------------+
| id|Name|Account balance|
+---+----+---------------+
|  1|  jk|        1000.00|
|  2|  pk|        2000.00|
|  3|  ak|        3000.00|
|  4|  ck|        4000.00|
+---+----+---------------+


In [26]:
input_df1=input_df.select(input_df["id"], input_df["Name"],encryptUDF(input_df["Account balance"]).alias("Account balance"))
#input_df1=input_df.select(input_df["id"], input_df["Name"],encryptUDF(input_df["Account balance"]))
input_df1.show(truncate=False)

+---+----+---------------+
|id |Name|Account balance|
+---+----+---------------+
|1  |jk  |[B@42d5698     |
|2  |pk  |[B@90f0844     |
|3  |ak  |[B@64bedae9    |
|4  |ck  |[B@2017bce5    |
+---+----+---------------+


#### How to Decrypt a dataframe column using UDF 

In [27]:
input_df1.select(input_df1["id"], input_df1["Name"],decryptUDF(input_df1["Account balance"]).alias("Account balance")).show(truncate=False)

+---+----+---------------+
|id |Name|Account balance|
+---+----+---------------+
|1  |jk  |1000.00        |
|2  |pk  |2000.00        |
|3  |ak  |3000.00        |
|4  |ck  |4000.00        |
+---+----+---------------+


#### UDF function might have performance impact. so be careful with large datasets